## 🚧 Preview Edition: Calling All Graphonauts! 🚀

**Attention brave Graphonauts!** You've been selected for an expedition into the uncharted territory of Custom Graphs. 

**What to expect on your journey:**
- 🐛 **Bugs**: They're not graph nodes, but they sure are connected to everything
- 🎢 **Breaking Changes**: The graph topology evolves faster than our docs
- 📝 **Incomplete Documentation**: We're mapping the terrain as we go
- 🔧 **Workarounds**: Sometimes the shortest path requires creative routing

**The Good News:** As a pioneering Graphonaut, you're helping us chart this new frontier! Your feedback will shape the future of graph exploration in Sentinel.

**Graphonaut Survival Kit:**
- ☕ Coffee (or your beverage of choice)
- 💾 Regular saves of your notebooks
- 🆘 Our support channel when edges don't connect as expected

---
*"In graphs we trust, in backups we believe."* 🛠️

**Ready to explore? Let's build some graphs!** 📊→🌌

## Check Pre-requisites
Before starting to use this notebook, please check the following.
- Ensure that you have the Security Operator role in the tenant - for the woodgrove tenant, you can go [here](https://aka.ms/woodgrove) and apply.
- Ensure that you are connecting to a region/account that has this functionality. To connect to the NOE region of Woodgrove, please follow instructions [here](https://microsoft-my.sharepoint.com/:w:/p/vamahtani/EVf-cJ5MJYVNifJ2QMJEo6QBiXKECGcSwixBIUbpxcyYNQ?e=RLB2FX).
- Ensure that you connect to a spark pool that has this feature enabled. Run the following cell to check if you are connected to a pool with the right module.

In [ ]:
# Ensure you are connected to the right environment
import pkg_resources
from packaging import version

min_version_for_this_notebook = version.parse("0.1.0")

def get_module_version(module_name) -> str:
    """Get version of a module using multiple methods"""
    try:
        version = pkg_resources.get_distribution(module_name).version
        #print(f"✅ {module_name}: {version}")
    except pkg_resources.DistributionNotFound:
        print(f"❌ pkg_resources: Package '{module_name}' not found")
        version = "not found"
    except Exception as e:
        version = "error"
    return version

graph_module = get_module_version('MicrosoftSentinelGraphProvider')
if graph_module == "not found" or graph_module == "error":
    print("❌ MicrosoftSentinelGraphProvider module is not installed. Please install it to proceed.")
    modules = ['MicrosoftSentinelProvider', 'MicrosoftSentinelGraphProvider']
    for module in modules:
        get_module_version(module)
else:
    print(f"MicrosoftSentinelGraphProvider module with version '{graph_module}' is installed.")
    if version.parse(graph_module) >= min_version_for_this_notebook:
        print(f"✅ Version {graph_module} will work with this notebook. You are good to go!")
    else:
        print(f"❌  Version {graph_module} is older than {min_version_for_this_notebook}. This notebook will NOT work.")
print("\nPlease share the following details when reporting any issues:")
print(f"region  : {spark.conf.get('spark.cluster.region')}")
print(f"account id : {spark.conf.get('spark.pjs.account.id')}")
print(f"pod id     : {spark.conf.get('spark.pjs.pod.id')}")
print(f"workspace name : {spark.conf.get('spark.synapse.workspace.name')}")
print(f"pool name : {spark.conf.get('spark.synapse.pool.name')}")


## 1️⃣ Define your Graph

In this step, we're defining the graph structure using the **GraphSpecBuilder** - think of it as your blueprint for turning security data into connected insights! 🎯

**What makes this cool?**

🔹 **Fluent API Design**: Chain methods together for readable, declarative graph definitions  
🔹 **Flexible Data Sources**: Build from existing Sentinel lake tables OR custom DataFrames  
🔹 **Automatic Type Inference**: The builder figures out data types - you focus on the relationships  
🔹 **Time-Aware Processing**: Filter your graph data by time ranges for focused analysis  
🔹 **Smart Schema Generation**: Automatically creates optimized node and edge schemas

**Key Concepts:**

- **Nodes** = Your entities (users, devices, IPs, etc.)
- **Edges** = The relationships between them (sign-in, access, communication, etc.)
- **Properties** = Attributes on nodes and edges (name, location, timestamp, etc.)

For starters, we'll build a simple security graph from existing Sentinel lake tables, showing the relationship between users and their devices through sign-in events. Later, you can create more complex graphs with custom data sources!

**Pro Tip:** Start simple with 2-3 entity types, then expand as you discover patterns! 🚀

Before starting, if you would like to control the verbosity of the output, please feel free to run the following optional cell with the updated verbosity level.

In [ ]:
# Skip this if you would like to go with the default output verbosity. If you want to control the logging verbosity levels, use the following.
import logging
import sentinel_graph

# Set the logging level for the entire package
logging.getLogger('sentinel_graph').setLevel(logging.INFO)  # or ERROR, WARNING, INFO, DEBUG. Default is INFO
print(f"Logging level set to: {logging.getLevelName(logging.getLogger('sentinel_graph').level)} and above")

In [ ]:
# create the graph using the graph spec builder
# type: ignore
# pyright: off
# pylance: off
from sentinel_graph.builders.graph_builder import GraphSpecBuilder

# lake tables we will use as input for the graph in this example
identity_info_tbl = "cgi_identity_info_SPRK"
device_info_tbl = "cgi_device_info_SPRK"
signin_logs_tbl = "cgi_signin_logs_SPRK"

my_graph = (GraphSpecBuilder.start()

        # Add user node from IdentityInfo table (using direct column names)
        .add_node("user") 
            .from_table(identity_info_tbl)
            .with_time_range(time_column="TimeGenerated", start_time="2025-10-22", end_time="2025-10-24")
            .with_columns("id", "name", "email", key="id", display="name")

        # Add device node from DeviceInfo table
        .add_node("device") 
            .from_table(device_info_tbl)
            .with_time_range(time_column="TimeGenerated", start_time="2025-10-22", end_time="2025-10-24")
            .with_columns("id", "name", "type", key="id", display="name")

        # Add edge between users and devices from SigninLogs table
        .add_edge("sign_in") 
            .from_table(signin_logs_tbl)
            .with_time_range(time_column="TimeGenerated", start_time="2025-10-22", end_time="2025-10-24")
            .source(id_column="UserId", node_type="user")
            .target(id_column="DeviceId", node_type="device")
            .with_columns("id", "location", key="id", display="id")
        
        ).done()

print(f"✅ Graph specification created successfully")



### 🎉 You now have a Graph Spec!

Congratulations Graphonaut! You've created your **GraphSpec** - your graph blueprint. Now you can:

- **Build & Materialize** - Execute ETL to create graph structures in the lake
- **Query with GQL** - Use Graph Query Language to traverse relationships
- **Analyze** - Run graph algorithms (PageRank, Connected Components, etc.)
- **Inspect & Visualize** - View schema and graph structure

## 2️⃣ 🔨 Build & Materialize Your Graph
Execute the ETL pipeline to transform your data into actual graph structures stored in the lake:

```python
result = my_graph.build_graph_with_data()
```

This creates optimized node and edge tables in your specified database, ready for querying and analysis. 


In [ ]:
# build your graph!

build_result = my_graph.build_graph_with_data()

print(f"Status: {build_result.get('status')}")


## 3️⃣ **Perform Graph Analytics**

You can now run your analytis on the Graph.

### 3️⃣.1️⃣ Blast Radius

Run Blast-Radius analysis from the source node to the target node.

In [ ]:
from sentinel_graph.builders.query_input import BlastRadiusQueryInput

result = my_graph.blast_radius(BlastRadiusQueryInput(source_property_value="user-003", target_property_value="device-003", min_hop_count=1))

result.show()

### 3️⃣.2️⃣ Centrality

In [ ]:
from sentinel_graph.builders.query_input import CentralityQueryInput

result = my_graph.centrality(CentralityQueryInput(
    participating_source_node_labels=["user", "device"], 
    participating_target_node_labels=["device", "user"], 
    participating_edge_labels=["sign_in"],
    is_directional=False)
    )

result.show()

### 3️⃣.3️⃣ Reachability

In [ ]:
from sentinel_graph.builders.query_input import ReachabilityQueryInput

result = my_graph.reachability(ReachabilityQueryInput(
    source_property_value="user-001",
    target_property_value="device-003"))

result.show()

### 3️⃣.4️⃣ K_Hop


In [ ]:
from sentinel_graph.builders.query_input import K_HopQueryInput

result = my_graph.k_hop(K_HopQueryInput(source_property_value="user-001"))

result.show()

### 3️⃣.5️⃣ Free form GQL Queries
Query your graph using Graph Query Language (GQL) - think SQL for graphs.

In [ ]:
query_result = my_graph.query("MATCH (u:user)-[s:sign_in]->(d:device) RETURN u, s, d LIMIT 10")
query_result.show()


### 3️⃣.6️⃣ Convert to Graphframe
You can also run more adhoc analytics by converting your graph to a graphframe.

In [ ]:
# convert to graph_frame 
gf = my_graph.to_graphframe()

# In-degree: How many edges coming INTO each node
print("\n=== In-Degree (Who gets connected to the most?) ===")
in_degrees = gf.inDegrees
in_degrees.orderBy("inDegree", ascending=False).show(10)

# PageRank - Find most "important" nodes
print("\n=== PageRank (Most Important Nodes) ===")
pagerank_results = gf.pageRank(resetProbability=0.15, maxIter=10)
pagerank_results.vertices.orderBy("pagerank", ascending=False).show(10)

## 4️⃣ **Inspect & Visualize**
You can inspect and visualize the graph spec

In [ ]:
# You'll need to have the latest plugin installed for this to work.
# This shows a sample of 100 elements from the graph as a visual. You can control the sample size with the limit parameter Ex: my_graph.show(limit=50).
my_graph.show()

## 🎉 Excellent Progress, Graphonaut!

You've now mastered the **core graph operations**:
- ✅ **Graph Building** - Defined and materialized your graph structure
- ✅ **Graph Analytics** - Ran algorithms like PageRank and degree analysis  
- ✅ **GQL Queries** - Explored relationships using Graph Query Language
- ✅ **Visualization** - Inspected your graph schema and structure

---

## 🚀 Additional Concepts & Advanced Techniques

The following sections explore **additional concepts** to enhance your graph building skills:
- Building graphs from **custom DataFrames** (instead of lake tables)

Let's dive deeper! 📊

### 7️⃣ Building Graphs from Custom DataFrames

Now let's explore an alternative approach: building graphs from **custom DataFrames** instead of lake tables. This gives you complete flexibility to:
- Pre-process and transform your data
- Combine multiple data sources
- Apply custom filtering and enrichment logic
- Test with synthetic or sample data

In this example, we'll create custom DataFrames with sample data and build a graph using the same user-device-signin pattern, but with the `.from_dataframe()` method instead of `.from_table()`.

#### 7️⃣.1️⃣ Create Sample DataFrames

First, we'll create sample DataFrames with synthetic security data. In a real scenario, you might:
- Load data from external sources (CSV, JSON, APIs)
- Query and transform existing lake tables
- Enrich data with lookups or joins
- Apply custom business logic

For this example, we're creating in-memory DataFrames with sample user identities, device information, and sign-in events.

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# create sample dataframes - 
print("Creating input dataframes...")

# Use current time for test data to work with date filtering
now = datetime.now()

# Create sample data with both expected and Sentinel-style columns
identity_info_data = [
# (id, name, email, TimeGenerated, UserId, UserDisplayName, UserPrincipalName)
("user-001", "Alice Smith", "alice@contoso.com", now - timedelta(hours=2), "user-001", "Alice Smith", "alice@contoso.com"),
("user-002", "Bob Johnson", "bob@contoso.com", now - timedelta(hours=1), "user-002", "Bob Johnson", "bob@contoso.com"),
("user-003", "Carol Davis", "carol@contoso.com", now - timedelta(minutes=30), "user-003", "Carol Davis", "carol@contoso.com"),
("user-004", "David Wilson", "david@contoso.com", now - timedelta(minutes=45), "user-004", "David Wilson", "david@contoso.com")
]

device_info_data = [
# (id, name, type, TimeGenerated, DeviceId, DeviceName, DeviceOS)
("device-001", "ALICE-LAPTOP", "Windows 11", now - timedelta(hours=2), "device-001", "ALICE-LAPTOP", "Windows 11"),
("device-002", "BOB-WORKSTATION", "Windows 10", now - timedelta(hours=1), "device-002", "BOB-WORKSTATION", "Windows 10"),
("device-003", "CAROL-MOBILE", "iOS 17", now - timedelta(minutes=30), "device-003", "CAROL-MOBILE", "iOS 17"),
("device-004", "DAVID-TABLET", "Android 13", now - timedelta(minutes=45), "device-004", "DAVID-TABLET", "Android 13")
]

signin_logs_data = [
# (id, location, TimeGenerated, SigninId, UserId, DeviceId)
("signin-001", "Seattle, WA", now - timedelta(minutes=45), "signin-001", "user-001", "device-001"),
("signin-002", "New York, NY", now - timedelta(minutes=30), "signin-002", "user-002", "device-002"),
("signin-003", "Austin, TX", now - timedelta(minutes=15), "signin-003", "user-003", "device-003"),
("signin-004", "Seattle, WA", now - timedelta(minutes=10), "signin-004", "user-001", "device-003"),  # Cross-device signin
("signin-005", "Denver, CO", now - timedelta(minutes=5), "signin-005", "user-004", "device-004")
]

# Define schemas that create the expected columns for graph processing
identity_info_schema = StructType([
StructField("id", StringType(), False),  # Expected by graph engine
StructField("name", StringType(), False),  # Expected by graph engine
StructField("email", StringType(), False),  # Expected by graph engine
StructField("TimeGenerated", TimestampType(), False),
# Keep original Sentinel columns for reference/relationships
StructField("UserId", StringType(), False),
StructField("UserDisplayName", StringType(), False),
StructField("UserPrincipalName", StringType(), False)
])

device_info_schema = StructType([
StructField("id", StringType(), False),  # Expected by graph engine
StructField("name", StringType(), False),  # Expected by graph engine
StructField("type", StringType(), False),  # Expected by graph engine
StructField("TimeGenerated", TimestampType(), False),
# Keep original Sentinel columns for reference/relationships
StructField("DeviceId", StringType(), False),
StructField("DeviceName", StringType(), False),
StructField("DeviceOS", StringType(), False)
])

signin_logs_schema = StructType([
StructField("id", StringType(), False),  # Expected by graph engine
StructField("location", StringType(), False),  # Expected by graph engine
StructField("TimeGenerated", TimestampType(), False),
# Keep original Sentinel columns for relationships (avoid duplicate location)
StructField("SigninId", StringType(), False),
StructField("UserId", StringType(), False),
StructField("DeviceId", StringType(), False)
])

# Create DataFrames with Sentinel table names
identity_info_df = spark.createDataFrame(identity_info_data, identity_info_schema)
device_info_df = spark.createDataFrame(device_info_data, device_info_schema)
signin_logs_df = spark.createDataFrame(signin_logs_data, signin_logs_schema)

print(f"   ✅ Created 'IdentityInfo' dataframe with {identity_info_df.count()} rows")
print(f"   ✅ Created 'DeviceInfo' dataframe with {device_info_df.count()} rows")
print(f"   ✅ Created 'SigninLogs' dataframe with {signin_logs_df.count()} rows")


#### 7️⃣.2️⃣ Build Graph from DataFrames

Now we'll build the graph using our custom DataFrames. Notice how the **same concepts and constructs** you used earlier with `.from_table()` apply seamlessly here with `.from_dataframe()`:

- Same `.add_node()` and `.add_edge()` fluent API
- Same `.with_columns()` for property selection
- Same `.source()` and `.target()` for edge definitions
- Same time-based filtering with `build_graph_with_data()`

**The only difference:** Instead of referencing table names, you pass DataFrame objects directly. This gives you complete control over data preparation while maintaining the same familiar graph building pattern.

In [ ]:
from sentinel_graph.builders.graph_builder import GraphSpecBuilder

# Define your graph using dataframes
my_graph = (GraphSpecBuilder.start()

            # Add user node from IdentityInfo table (using direct column names)
            .add_node("user")
                .from_dataframe(identity_info_df)
                .with_columns("id", "name", "email", key="id", display="name")

            # Add device node from DeviceInfo table
            .add_node("device")
                .from_dataframe(device_info_df)
                .with_columns("id", "name", "type", key="id", display="name")

            # Add edge between users and devices from SigninLogs table
            .add_edge("sign_in")
                .from_dataframe(signin_logs_df)
                .source(id_column="UserId", node_type="user")
                .target(id_column="DeviceId", node_type="device")
                .with_columns("id", "location", key="id", display="id")
        ).done()
    
print(f"✅ Graph specification created successfully")


In [ ]:
# Now build it with data
print("\n Executing ETL pipeline...")
build_result = my_graph.build_graph_with_data()

if build_result.get("status") != "success":
    raise Exception("Graph build from dataframes failed.")
else:
    print("✅ Graph built successfully from dataframes.")

#### 7️⃣.3️⃣ Analytics & Queries Work Exactly the Same

**Great news!** All the analytics and query operations you learned earlier work identically with this DataFrame-based graph:

✅ **Graph Analytics** - `.to_graphframe()`, PageRank, Connected Components, etc.  
✅ **GQL Queries** - `.query()` with the same GQL syntax  
✅ **Inspection** - `.show()`, schema viewing  
✅ **Visualization** - Same visualization functions apply

Whether your graph comes from lake tables or custom DataFrames, the graph operations are completely unified. You can use the same `my_graph` object with all the methods from sections 3-6 above.

**Try it yourself:** Run analytics or queries on `my_graph` using the exact same code from earlier sections!

In [ ]:
print("\nRunning sample query to verify graph contents...")
query_result = my_graph.query("MATCH (u:user)-[s:sign_in]->(d:device) RETURN u,s, d LIMIT 10")

query_result.show()


In [ ]:
# Here is your homework Graphonaut! Explore the Graph!



# 🎓 Congratulations, Graphonaut!

You've successfully navigated your first Custom Graph expedition! 🚀

## What You've Accomplished

✅ **Defined a Graph Spec** - Created your graph blueprint with nodes and edges  
✅ **Built & Materialized** - Executed ETL to transform data into graph structures  
✅ **Ran Analytics** - Used GraphFrames for PageRank and graph algorithms  
✅ **Queried with GQL** - Explored relationships using Graph Query Language  
✅ **Visualized** - Inspected your graph structure and schema

## Next Steps for Your Graph Journey

### 🔍 Explore More
- Bring your interesting Security scenario and try it out!
- Add additional entity types (IPs, applications, files, etc.)
- Create more complex relationships (multi-hop patterns)
- Experiment with different time ranges and filters

### 📊 Advanced Analytics
- Try Connected Components to find isolated security clusters
- Use Shortest Paths to trace attack chains
- Implement custom graph algorithms for threat detection

### 🚀 Share & Collaborate
- Export your graph for use in other tools
- Share insights with your security team
- Contribute feedback to improve the platform


*"The graph is not just data—it's a map of relationships waiting to reveal insights."* 🗺️✨

**Keep exploring, Graphonaut!** 🌌